# Data-Efficient Multilingual Vision-Language Modeling for Low-Resource Languages

This project investigates parameter-efficient adaptation of a pretrained
vision-language model for low-resource Tigrinya generation.

**Research question:** How effectively can a pretrained multilingual
vision-language model be adapted to Tigrinya with limited multimodal
supervision, and how do multimodal input and training-data size affect
generation quality?

The experiments use Gemma 3 4B Instruct and the Tigrinya subset of the LaMuN
dataset. The study compares zero-shot generation, multimodal QLoRA, text-only
QLoRA, and multimodal training with different data sizes.

> **Note:** This notebook contains the complete experimental workflow
> (methods, code, evaluation). See `README.md` for the research summary,
> polished results tables, findings, and reproducibility instructions.


## 1. Environment Setup

In [10]:
!nvidia-smi

Fri Sep 11 20:31:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             87W /  600W |    8871MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

> **Environment note:** the following installation steps were used to ensure
> compatibility with the specific Colab runtime this experiment was run on.

In [11]:
!pip uninstall -y pillow -q
!pip install --no-cache-dir --force-reinstall "pillow==11.1.0" -q
!pip install -U transformers accelerate sentencepiece datasets -q
!pip install -U bitsandbytes peft trl accelerate -q
!pip install -q sacrebleu rouge-score


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 82.7 MB/s eta 0:00:00


In [12]:
# Core
import os
import gc
import json
import time
import random
import re
from dataclasses import dataclass
from typing import List
from collections import Counter

# Data
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

# Models
from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Gemma3ForConditionalGeneration,
)

from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# Evaluation
from sacrebleu import corpus_bleu, corpus_chrf, sentence_bleu, sentence_chrf
from rouge_score import rouge_scorer

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [13]:
# =========================
# Experiment configuration
# =========================
# Used throughout the notebook instead of repeating hard-coded numbers.

MODEL_ID = "google/gemma-3-4b-it"
DATASET_ID = "tharindu/LaMuN"
LANGUAGE_CONFIG = "tir"

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# Quantization
LOAD_IN_4BIT = True
BNB_QUANT_TYPE = "nf4"
BNB_USE_DOUBLE_QUANT = True

# Training
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_GRAD_NORM = 1.0

# An initial full-data exploratory run over NUM_EPOCHS_EXPLORATORY epochs
# showed validation loss increasing after epoch 1 (overfitting). The
# controlled ablations below therefore use a fixed NUM_EPOCHS_ABLATION = 1
# training budget across all conditions/data sizes for a fair comparison.
NUM_EPOCHS_EXPLORATORY = 3
NUM_EPOCHS_ABLATION = 1

# Generation
MAX_NEW_TOKENS = 80

# Data efficiency
DATA_SIZES = [100, 500, 1000, 2440]

print("Model:", MODEL_ID)
print("Dataset:", DATASET_ID, "/", LANGUAGE_CONFIG)


Model: google/gemma-3-4b-it
Dataset: tharindu/LaMuN / tir


## 2. Base Model (zero-shot)

> **Authentication:** `google/gemma-3-4b-it` is a gated model. To run this
> notebook yourself, authenticate with your own Hugging Face token first
> (e.g. `huggingface_hub.login()` in your own environment) before running
> the cells below.

In [ ]:
import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

model_id = "google/gemma-3-4b-it"

processor = AutoProcessor.from_pretrained(model_id)

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()

print("Model loaded successfully.")

**Zero-shot diagnostic:** preliminary tests showed substantially weaker Tigrinya
generation than English generation, motivating parameter-efficient adaptation
rather than pure zero-shot prompting.

## 3. Dataset

In [16]:
from datasets import load_dataset

dataset = load_dataset(
    "tharindu/LaMuN",
    "tir",
    split="train",
    streaming=True
)

print(dataset)

README.md:   0%|          | 0.00/41.4k [00:00<?, ?B/s]

IterableDataset({
    features: ['image', 'caption', 'title', 'content', 'news_source', 'language'],
    num_shards: 1
})


In [17]:
count = 0

for example in dataset:
    count += 1

print("Number of Tigrinya training examples:", count)

Number of Tigrinya training examples: 3028


In [18]:
articles = []

for example in dataset:
    articles.append({
        "content": example["content"],
        "title": example["title"],
        "news_source": example["news_source"],
        "caption": example["caption"],
    })

print("Examples collected:", len(articles))

Examples collected: 3028


In [19]:
import pandas as pd

df = pd.DataFrame(articles)

print("Total examples:", len(df))
print("Unique content:", df["content"].nunique())
print("Unique titles:", df["title"].nunique())

Total examples: 3028
Unique content: 1768
Unique titles: 1778


In [20]:
print("Articles with >1 image:", (df["content"].value_counts() > 1).sum())
print("Articles with >5 images:", (df["content"].value_counts() > 5).sum())
print("Articles with >10 images:", (df["content"].value_counts() > 10).sum())
print("Articles with >20 images:", (df["content"].value_counts() > 20).sum())

Articles with >1 image: 589
Articles with >5 images: 40
Articles with >10 images: 4
Articles with >20 images: 2


### 3.1 Article-level train/validation/test split

Splitting is done at the *article* level (not the example level) so that no article's images leak across splits.

In [21]:
from sklearn.model_selection import train_test_split

# Get one row per unique article
articles_df = (
    df[["content", "title", "news_source"]]
    .drop_duplicates(subset="content")
    .reset_index(drop=True)
)

print("Unique articles:", len(articles_df))

# 80% train, 20% temporary
train_articles, temp_articles = train_test_split(
    articles_df,
    test_size=0.20,
    random_state=42
)

# Split the remaining 20% into validation and test
val_articles, test_articles = train_test_split(
    temp_articles,
    test_size=0.50,
    random_state=42
)

print("Train articles:", len(train_articles))
print("Validation articles:", len(val_articles))
print("Test articles:", len(test_articles))

Unique articles: 1768
Train articles: 1414
Validation articles: 177
Test articles: 177


In [22]:
# Create article → split mapping

train_content = set(train_articles["content"])
val_content = set(val_articles["content"])
test_content = set(test_articles["content"])

def assign_split(content):
    if content in train_content:
        return "train"
    elif content in val_content:
        return "validation"
    elif content in test_content:
        return "test"
    else:
        return "unknown"

df["split"] = df["content"].apply(assign_split)

print(df["split"].value_counts())

split
train         2440
validation     300
test           288
Name: count, dtype: int64


**Reproducibility check — verifying zero split leakage:**

In [23]:
print("Train articles:", df[df["split"] == "train"]["content"].nunique())
print("Validation articles:", df[df["split"] == "validation"]["content"].nunique())
print("Test articles:", df[df["split"] == "test"]["content"].nunique())

print("\nArticle overlap checks:")

print(
    "Train ∩ Validation:",
    len(train_content & val_content)
)

print(
    "Train ∩ Test:",
    len(train_content & test_content)
)

print(
    "Validation ∩ Test:",
    len(val_content & test_content)
)

Train articles: 1414
Validation articles: 177
Test articles: 177

Article overlap checks:
Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [24]:
df.to_csv("lamun_tigrinya_article_split.csv", index=False)

print("Saved: lamun_tigrinya_article_split.csv")

Saved: lamun_tigrinya_article_split.csv


In [ ]:
from datasets import load_dataset

lamun_tir = load_dataset(
    "tharindu/LaMuN",
    "tir",
    split="train"
)

print(lamun_tir)
print(lamun_tir.column_names)

In [26]:
# Create a mapping from article content → split
split_map = {}

for content in train_content:
    split_map[content] = "train"

for content in val_content:
    split_map[content] = "validation"

for content in test_content:
    split_map[content] = "test"


# Add split to the Hugging Face dataset
def add_split(example):
    example["split"] = split_map[example["content"]]
    return example

lamun_tir = lamun_tir.map(add_split)

print(lamun_tir)
print(lamun_tir["split"][:10])

Map:   0%|          | 0/3028 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'caption', 'title', 'content', 'news_source', 'language', 'split'],
    num_rows: 3028
})
['train', 'train', 'train', 'train', 'train', 'train', 'train', 'train', 'train', 'train']


In [27]:
from collections import Counter

counts = Counter(lamun_tir["split"])

print(counts)

Counter({'train': 2440, 'validation': 300, 'test': 288})


In [28]:
train_ds = lamun_tir.filter(lambda x: x["split"] == "train")
val_ds   = lamun_tir.filter(lambda x: x["split"] == "validation")
test_ds  = lamun_tir.filter(lambda x: x["split"] == "test")

print("Train:", len(train_ds))
print("Validation:", len(val_ds))
print("Test:", len(test_ds))

Filter:   0%|          | 0/3028 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3028 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3028 [00:00<?, ? examples/s]

Train: 2440
Validation: 300
Test: 288


### 3.2 Dataset inspection

In [29]:
import pandas as pd

# Convert the Hugging Face dataset to a metadata DataFrame
analysis_df = lamun_tir.remove_columns(["image"]).to_pandas()

# Basic statistics
print("Total examples:", len(analysis_df))
print("Unique articles:", analysis_df["content"].nunique())

print("\nExamples by source:")
print(analysis_df["news_source"].value_counts())

print("\nCaption length (characters):")
print(analysis_df["caption"].str.len().describe())

print("\nTitle length (characters):")
print(analysis_df["title"].str.len().describe())

print("\nArticle length (characters):")
print(analysis_df["content"].str.len().describe())

Total examples: 3028
Unique articles: 1768

Examples by source:
news_source
BBC    2553
VOA     475
Name: count, dtype: int64

Caption length (characters):
count    3028.000000
mean       46.519155
std        33.900854
min         1.000000
25%        27.000000
50%        42.000000
75%        59.000000
max       525.000000
Name: caption, dtype: float64

Title length (characters):
count    3028.000000
mean       42.614597
std        10.688076
min        12.000000
25%        35.000000
50%        43.000000
75%        50.000000
max        92.000000
Name: title, dtype: float64

Article length (characters):
count     3028.000000
mean      3566.480185
std       2450.713272
min          0.000000
25%       1619.750000
50%       3352.000000
75%       5049.000000
max      20030.000000
Name: content, dtype: float64


In [30]:
print("\nExamples where title == caption:")
print(
    (analysis_df["title"].str.strip() == analysis_df["caption"].str.strip()).sum()
)

print("\nPercentage where title == caption:")
print(
    100 * (
        analysis_df["title"].str.strip() == analysis_df["caption"].str.strip()
    ).mean()
)


Examples where title == caption:
34

Percentage where title == caption:
1.1228533685601056


## 4. Zero-Shot Baseline

In [31]:
import torch

def generate_tigrinya(image, prompt, max_new_tokens=80):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = output[:, inputs["input_ids"].shape[1]:]

    return processor.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

In [32]:
def generate_text_only(prompt, max_new_tokens=80):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt}
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = output[:, inputs["input_ids"].shape[1]:]

    return processor.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )


def generate_image_text(image, prompt, max_new_tokens=80):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = output[:, inputs["input_ids"].shape[1]:]

    return processor.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

In [33]:
for i in range(5):

    example = test_ds[i]

    title = example["title"]
    image = example["image"]
    reference = example["caption"]

    text_prompt = f"""
Based on the following Tigrinya news title, write a brief factual
caption in Tigrinya.

Title: {title}
"""

    image_prompt = """
Describe the image briefly and factually in Tigrinya.
Do not invent information that cannot be determined from the image.
"""

    multimodal_prompt = f"""
Look at the image and use the following Tigrinya news title as context.
Write a brief, factual caption in Tigrinya describing what is shown.
Do not invent information that is not supported by the image or title.

Title: {title}
"""

    text_output = generate_text_only(text_prompt)

    image_output = generate_image_text(
        image,
        image_prompt
    )

    multimodal_output = generate_image_text(
        image,
        multimodal_prompt
    )

    print("=" * 100)
    print(f"EXAMPLE {i+1}")

    print("\nREFERENCE:")
    print(reference)

    print("\nTEXT ONLY:")
    print(text_output)

    print("\nIMAGE ONLY:")
    print(image_output)

    print("\nIMAGE + TITLE:")
    print(multimodal_output)

EXAMPLE 1

REFERENCE:
ንሰለስተ ኣዋርሕ ጅሆ ተታሒዛ ብተደጋጋሚ ብወተሃደራት ተዓሚፃ።

TEXT ONLY:
Here are a few options for a brief factual caption in Tigrinya, based on the title, with varying levels of detail:

**Option 1 (Most Concise):**

"ትግራይ ክልል፡ 'ምፅናት ዓሌት' ብፁም ፀብፃብ መርመራ ተጀማሪ።"
(Tigray region: Investigation into '

IMAGE ONLY:
Here's a brief, factual description of the image in Tigrinya:

ይང་རྩ་མོ་གསེར་གྱི་སྤྲུལ་སྐྱེས་གསེར་གྱི་གདུང་གཟིངས་བཞག་པའི་སྤྲུལ་སྐྱེས་གཞན་གྱི་གདུང་གཟིངས་བཞ

IMAGE + TITLE:
Here's a brief, factual caption in Tigrinya describing the image, based on the provided title:

"ብምፅናት ዓሌት ብሓቂ ሓድሽ ፀብፃብ መርመራ እዩ ዝ Conduct እዮም፤ ኣብዚ ፎቶ ዝተ显የ ሰንበት ብሓቂ ሓንቲ ሮዝ ዝተቐ
EXAMPLE 2

REFERENCE:
ዋና ኣካያዲት ስራሕ ዓለምለኸ ማዕከን ገንዘብ (IMF) ክሪስታሊና ጆርጂየቫ

TEXT ONLY:
Here are a few options for a brief factual caption in Tigrinya, based on the title, with varying levels of detail:

**Option 1 (Most Concise):**

"ኢትዮጵያ ኣብ ዓለምለኸ ማዕከን ገንዘብ ይውእዱ፡ ብኣዒንቲ ፋይናንሳዊ ጕዕዞ።"
(Ethiop

IMAGE ONLY:
Here's a brief and factual description of

In [34]:
import re
import pandas as pd

def script_stats(text):
    if not text:
        return {
            "length": 0,
            "tigrinya_pct": 0,
            "latin_pct": 0,
            "other_pct": 0,
            "tigrinya_chars": 0,
            "latin_chars": 0,
            "other_chars": 0,
        }

    # Ethiopic Unicode block
    tigrinya_chars = len(re.findall(r'[\u1200-\u137F]', text))

    # Latin alphabet
    latin_chars = len(re.findall(r'[A-Za-z]', text))

    # Count alphabetic/symbolic characters excluding whitespace
    non_space = len(re.findall(r'\S', text))

    other_chars = max(non_space - tigrinya_chars - latin_chars, 0)

    return {
        "length": len(text),
        "tigrinya_pct": round(100 * tigrinya_chars / non_space, 2) if non_space else 0,
        "latin_pct": round(100 * latin_chars / non_space, 2) if non_space else 0,
        "other_pct": round(100 * other_chars / non_space, 2) if non_space else 0,
        "tigrinya_chars": tigrinya_chars,
        "latin_chars": latin_chars,
        "other_chars": other_chars,
    }

### 4.1 Full baseline generation and evaluation (test set, n=288)

In [35]:
import json
import os
from tqdm.auto import tqdm

baseline_results = []

for i in tqdm(range(len(test_ds))):

    example = test_ds[i]

    title = example["title"]
    image = example["image"]
    reference = example["caption"]

    text_prompt = f"""
Based on the following Tigrinya news title, write a brief factual
caption in Tigrinya.

Title: {title}
"""

    image_prompt = """
Describe the image briefly and factually in Tigrinya.
Do not invent information that cannot be determined from the image.
"""

    multimodal_prompt = f"""
Look at the image and use the following Tigrinya news title as context.
Write a brief, factual caption in Tigrinya describing what is shown.
Do not invent information that is not supported by the image or title.

Title: {title}
"""

    text_output = generate_text_only(
        text_prompt,
        max_new_tokens=80
    )

    image_output = generate_image_text(
        image,
        image_prompt,
        max_new_tokens=80
    )

    multimodal_output = generate_image_text(
        image,
        multimodal_prompt,
        max_new_tokens=80
    )

    baseline_results.append({
        "id": i,
        "reference": reference,
        "title": title,
        "text_only": text_output,
        "image_only": image_output,
        "image_title": multimodal_output
    })

  0%|          | 0/288 [00:00<?, ?it/s]

In [36]:
with open(
    "/content/lamun_tigrinya_zero_shot_baseline.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        baseline_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", len(baseline_results), "examples")

Saved: 288 examples


**Raw metrics (primary result):**

In [37]:
from sacrebleu import corpus_bleu, corpus_chrf
from rouge_score import rouge_scorer

conditions = ["text_only", "image_only", "image_title"]

references = [
    result["reference"]
    for result in baseline_results
]

metric_results = []

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

for condition in conditions:

    predictions = [
        result[condition]
        for result in baseline_results
    ]

    bleu = corpus_bleu(
        predictions,
        [references]
    )

    chrf = corpus_chrf(
        predictions,
        [references]
    )

    rouge_scores = []

    for pred, ref in zip(predictions, references):
        score = rouge.score(ref, pred)["rougeL"].fmeasure
        rouge_scores.append(score)

    metric_results.append({
        "condition": condition,
        "BLEU": bleu.score,
        "chrF": chrf.score,
        "ROUGE-L": sum(rouge_scores) / len(rouge_scores)
    })

metrics_df = pd.DataFrame(metric_results)

metrics_df

,condition,BLEU,chrF,ROUGE-L
0,text_only,0.281270,6.710586,0.002778
1,image_only,0.006788,1.939094,0.002434
2,image_title,0.094597,3.882567,0.001657


**Generation diagnostics** (script composition, preamble rate):

In [38]:
diagnostic_rows = []

for result in baseline_results:

    for condition in ["text_only", "image_only", "image_title"]:

        stats = script_stats(result[condition])

        diagnostic_rows.append({
            "id": result["id"],
            "condition": condition,
            **stats
        })

baseline_diagnostic = pd.DataFrame(diagnostic_rows)

baseline_summary = (
    baseline_diagnostic
    .groupby("condition")
    [["length", "tigrinya_pct", "latin_pct", "other_pct"]]
    .agg(["mean", "median", "std"])
    .round(2)
)

baseline_summary

length               tigrinya_pct               latin_pct         \
               mean median    std         mean median    std      mean median   
condition                                                                       
image_only   170.89  162.5  31.60        23.77  23.14  17.23     48.47  47.17   
image_title  208.00  208.0  28.77        17.60  18.56  12.58     63.92  63.13   
text_only    217.33  217.5  25.36        22.49  21.37   7.48     68.51  70.09   

                  other_pct                
              std      mean median    std  
condition                                  
image_only   9.69     27.75  20.90  21.02  
image_title  8.18     18.48  11.26  14.29  
text_only    7.83      9.00   8.85   2.19

In [41]:
preamble_patterns = [
    r"here'?s",
    r"here are",
    r"brief",
    r"factual caption",
    r"description of the image",
    r"based on the title",
    r"translation",
    r"option 1",
    r"option 2",
]

def has_preamble(text):
    text_lower = text.lower()
    return any(re.search(pattern, text_lower) for pattern in preamble_patterns)

In [42]:
for condition in ["text_only", "image_only", "image_title"]:

    rate = (
        baseline_diagnostic[
            baseline_diagnostic["condition"] == condition
        ]["id"]
        .count()
    )

    preamble_count = sum(
        has_preamble(result[condition])
        for result in baseline_results
    )

    print(
        f"{condition}: "
        f"{preamble_count}/{len(baseline_results)} "
        f"({100*preamble_count/len(baseline_results):.2f}%)"
    )

text_only: 288/288 (100.00%)
image_only: 288/288 (100.00%)
image_title: 288/288 (100.00%)


**Post-hoc normalization (diagnostic, not the primary result).** Gemma
frequently prepended English instruction-following preamble (e.g. *"Here are a
few options..."*) before the Tigrinya text. We strip this boilerplate without
altering the actual generated Tigrinya content, and recompute metrics on the
normalized text purely as a secondary check — the raw metrics above remain the
reported result.

In [43]:
import re

def normalize_prediction(text):
    """
    Remove obvious instruction-following boilerplate without
    changing the model's actual Tigrinya content.
    """
    text = text.strip()

    # Remove common English preambles
    patterns = [
        r"^Here are a few options.*?:\s*",
        r"^Here is a brief.*?:\s*",
        r"^Here's a brief.*?:\s*",
        r"^A brief factual caption.*?:\s*",
        r"^Option\s+\d+\s*\(.*?\):\s*",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE | re.DOTALL)

    # Remove leading/trailing quotation marks
    text = text.strip().strip('"').strip("'").strip()

    return text

In [44]:
for result in baseline_results:
    result["text_only_norm"] = normalize_prediction(result["text_only"])
    result["image_only_norm"] = normalize_prediction(result["image_only"])
    result["image_title_norm"] = normalize_prediction(result["image_title"])

In [45]:
from sacrebleu import corpus_bleu, corpus_chrf
from rouge_score import rouge_scorer
import pandas as pd

conditions = ["text_only", "image_only", "image_title"]

references = [r["reference"] for r in baseline_results]

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

metric_results = []

for condition in conditions:

    predictions = [
        r[f"{condition}_norm"]
        for r in baseline_results
    ]

    bleu = corpus_bleu(
        predictions,
        [references]
    )

    chrf = corpus_chrf(
        predictions,
        [references]
    )

    rouge_scores = []

    for pred, ref in zip(predictions, references):
        score = rouge.score(ref, pred)["rougeL"].fmeasure
        rouge_scores.append(score)

    metric_results.append({
        "condition": condition,
        "BLEU": bleu.score,
        "chrF": chrf.score,
        "ROUGE-L": sum(rouge_scores) / len(rouge_scores)
    })

normalized_metrics_df = pd.DataFrame(metric_results)

normalized_metrics_df

,condition,BLEU,chrF,ROUGE-L
0,text_only,0.423309,6.371951,0.005622
1,image_only,0.009097,2.265200,0.001389
2,image_title,0.139185,4.983863,0.006076


In [46]:
print("RAW")
display(metrics_df)

print("\nNORMALIZED")
display(normalized_metrics_df)

RAW


,condition,BLEU,chrF,ROUGE-L
0,text_only,0.281270,6.710586,0.002778
1,image_only,0.006788,1.939094,0.002434
2,image_title,0.094597,3.882567,0.001657



NORMALIZED


,condition,BLEU,chrF,ROUGE-L
0,text_only,0.423309,6.371951,0.005622
1,image_only,0.009097,2.265200,0.001389
2,image_title,0.139185,4.983863,0.006076


## 5. Multimodal QLoRA Fine-Tuning

In [47]:
def make_training_messages(example):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": example["image"],
                },
                {
                    "type": "text",
                    "text": (
                        "ዕላማ ዜና ተመርኲስካ፣ ነቲ ስእሊ ዝገልጽ "
                        "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                        f"ኣርእስቲ ዜና፦ {example['title']}"
                    ),
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": example["caption"],
                }
            ],
        },
    ]

**Implementation note.** Preprocessing `pixel_values` inside `dataset.map()` caused
Arrow serialization overflow on this dataset. We instead move all image/text
preprocessing into the data collator (applied per-batch, not cached to disk), which
resolved the issue and is the approach used below.

In [48]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()

print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

GPU memory allocated: 0.01 GB


In [49]:
from transformers import BitsAndBytesConfig, Gemma3ForConditionalGeneration

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

print("Model loaded successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Model loaded successfully.


In [50]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


In [51]:
from dataclasses import dataclass
from typing import List
import torch


def make_training_messages(example):
    return [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": example["image"]},
                {
                    "type": "text",
                    "text": (
                        "ዕላማ ዜና ተመርኲስካ፣ ነቲ ስእሊ ዝገልጽ "
                        "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                        f"ኣርእስቲ ዜና፦ {example['title']}"
                    ),
                },
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["caption"]}
            ],
        },
    ]


@dataclass
class Gemma3MultimodalCollator:

    processor: object

    def __call__(self, examples: List[dict]):

        full_inputs_list = []
        prompt_inputs_list = []

        # --------------------------------------------------
        # Process each example
        # --------------------------------------------------
        for example in examples:

            messages = make_training_messages(example)

            # Full conversation = user + target caption
            full_inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=False,
                tokenize=True,
                return_tensors="pt",
                return_dict=True,
            )

            # Prompt only = user message
            prompt_inputs = self.processor.apply_chat_template(
                messages[:1],
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt",
                return_dict=True,
            )

            full_inputs_list.append(full_inputs)
            prompt_inputs_list.append(prompt_inputs)

        # --------------------------------------------------
        # Pad text sequences
        # --------------------------------------------------

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [
                x["input_ids"][0]
                for x in full_inputs_list
            ],
            batch_first=True,
            padding_value=self.processor.tokenizer.pad_token_id,
        )

        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [
                x["attention_mask"][0]
                for x in full_inputs_list
            ],
            batch_first=True,
            padding_value=0,
        )

        token_type_ids = torch.nn.utils.rnn.pad_sequence(
            [
                x["token_type_ids"][0]
                for x in full_inputs_list
            ],
            batch_first=True,
            padding_value=0,
        )

        # --------------------------------------------------
        # Construct labels
        # --------------------------------------------------

        labels = []

        for full_inputs, prompt_inputs in zip(
            full_inputs_list,
            prompt_inputs_list
        ):

            ids = full_inputs["input_ids"][0].clone()

            prompt_length = prompt_inputs["input_ids"].shape[1]

            # Ignore everything belonging to the prompt
            ids[:prompt_length] = -100

            labels.append(ids)

        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        # --------------------------------------------------
        # Stack images
        # --------------------------------------------------

        pixel_values = torch.stack(
            [
                x["pixel_values"][0]
                for x in full_inputs_list
            ]
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "pixel_values": pixel_values,
            "labels": labels,
        }

In [52]:
import torch

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

In [55]:
collator = Gemma3MultimodalCollator(processor)

In [57]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=2,
    pin_memory=True,
)

In [58]:
num_epochs = NUM_EPOCHS_EXPLORATORY

gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS

max_grad_norm = MAX_GRAD_NORM

save_dir = "/content/gemma3_tigrinya_qlora"

In [59]:
@torch.no_grad()
def evaluate_loss(model, val_loader):

    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in tqdm(
        val_loader,
        desc="Validation"
    ):

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):
            outputs = model(**batch)

        total_loss += outputs.loss.item()
        total_batches += 1

    return total_loss / total_batches

In [60]:
import os
import time
from tqdm.auto import tqdm

os.makedirs(save_dir, exist_ok=True)

best_val_loss = float("inf")
global_step = 0

training_history = []

model.train()

for epoch in range(num_epochs):

    epoch_start = time.time()

    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{num_epochs}"
    )

    for step, batch in enumerate(progress):

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        # -----------------------------
        # Forward pass
        # -----------------------------
        with torch.autocast(
            device_type="cuda",
            dtype=torch.bfloat16
        ):
            outputs = model(**batch)
            loss = outputs.loss

        # -----------------------------
        # Gradient accumulation
        # -----------------------------
        loss_for_backward = (
            loss / gradient_accumulation_steps
        )

        loss_for_backward.backward()

        running_loss += loss.item()

        # -----------------------------
        # Optimizer step
        # -----------------------------
        if (
            (step + 1) % gradient_accumulation_steps == 0
            or (step + 1) == len(train_loader)
        ):

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_grad_norm
            )

            optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            global_step += 1

            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                step=global_step
            )

    # -----------------------------
    # Epoch training loss
    # -----------------------------
    train_loss = running_loss / len(train_loader)

    # -----------------------------
    # Validation
    # -----------------------------
    val_loss = evaluate_loss(
        model,
        val_loader
    )

    epoch_time = time.time() - epoch_start

    print()
    print("=" * 60)
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train loss: {train_loss:.4f}")
    print(f"Val loss:   {val_loss:.4f}")
    print(f"Time:       {epoch_time / 60:.2f} minutes")
    print("=" * 60)

    training_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "global_step": global_step,
        "time_minutes": epoch_time / 60,
    })

    # -----------------------------
    # Save best adapter
    # -----------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_dir = os.path.join(
            save_dir,
            "best_adapter"
        )

        model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)

        print(
            f"✓ New best model saved "
            f"(val loss = {val_loss:.4f})"
        )

Epoch 1/3:   0%|          | 0/1220 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Validation:   0%|          | 0/150 [00:00<?, ?it/s]


Epoch 1/3
Train loss: 2.7379
Val loss:   2.5100
Time:       10.85 minutes
✓ New best model saved (val loss = 2.5100)


Epoch 2/3:   0%|          | 0/1220 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]


Epoch 2/3
Train loss: 2.0826
Val loss:   2.4232
Time:       8.29 minutes
✓ New best model saved (val loss = 2.4232)


Epoch 3/3:   0%|          | 0/1220 [00:00<?, ?it/s]

Validation:   0%|          | 0/150 [00:00<?, ?it/s]


Epoch 3/3
Train loss: 1.6334
Val loss:   2.4592
Time:       8.30 minutes


**Training protocol note.** This exploratory run over `NUM_EPOCHS_EXPLORATORY`
(3) epochs showed validation loss improving through epoch 2 and then increasing
in epoch 3, indicating mild overfitting. Validation loss by epoch was
2.5100 → 2.4232 → 2.4592. Based on this behavior, the controlled ablations in
Sections 6–7 use a fixed `NUM_EPOCHS_ABLATION = 1` training budget across
conditions and data sizes for a controlled comparison.

### 5.1 Evaluation on test set

In [61]:
from transformers import Gemma3ForConditionalGeneration
from peft import PeftModel
import torch

# Reload quantized base model
base_model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load BEST LoRA adapter — Epoch 1
best_model = PeftModel.from_pretrained(
    base_model,
    "/content/gemma3_tigrinya_qlora/best_adapter",
)

best_model.eval()

print("Best validation loss:", best_val_loss)
print("Best model loaded successfully.")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Best validation loss: 2.4231503280003865
Best model loaded successfully.


In [62]:
def generate_with_best_model(image, title, max_new_tokens=80):

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {
                    "type": "text",
                    "text": (
                        "ዕላማ ዜና ተመርኲስካ፣ ነቲ ስእሊ ዝገልጽ "
                        "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                        f"ኣርእስቲ ዜና፦ {title}"
                    ),
                },
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(best_model.device)

    with torch.inference_mode():

        output = best_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = output[
        :, inputs["input_ids"].shape[1]:
    ]

    return processor.decode(
        generated_tokens[0],
        skip_special_tokens=True,
    )

In [63]:
import json
import time
from tqdm.auto import tqdm

test_results = []

start_time = time.time()

for i, example in enumerate(
    tqdm(test_ds, desc="Evaluating QLoRA on test set")
):

    prediction = generate_with_best_model(
        example["image"],
        example["title"],
        max_new_tokens=80,
    )

    test_results.append({
        "test_index": i,
        "title": example["title"],
        "reference": example["caption"],
        "prediction": prediction,
        "news_source": example["news_source"],
    })

elapsed = time.time() - start_time

output_path = "/content/lamun_tigrinya_qlora_test_predictions.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        test_results,
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"\nSaved: {output_path}")
print(f"Examples: {len(test_results)}")
print(f"Time: {elapsed / 60:.2f} minutes")

Evaluating QLoRA on test set:   0%|          | 0/288 [00:00<?, ?it/s]


Saved: /content/lamun_tigrinya_qlora_test_predictions.json
Examples: 288
Time: 4.85 minutes


In [64]:
import json
import re
import numpy as np
from collections import Counter

# Load QLoRA predictions
with open(
    "/content/lamun_tigrinya_qlora_test_predictions.json",
    "r",
    encoding="utf-8"
) as f:
    qlora_results = json.load(f)


def script_stats(text):
    if not text:
        return {
            "length": 0,
            "tigrinya_pct": 0,
            "latin_pct": 0,
            "other_pct": 0,
        }

    chars = [c for c in text if not c.isspace()]
    n = len(chars)

    tigrinya = sum(
        "\u1200" <= c <= "\u137f"
        for c in chars
    )

    latin = sum(
        ("A" <= c <= "Z") or
        ("a" <= c <= "z")
        for c in chars
    )

    other = n - tigrinya - latin

    return {
        "length": n,
        "tigrinya_pct": 100 * tigrinya / n,
        "latin_pct": 100 * latin / n,
        "other_pct": 100 * other / n,
    }


stats = []

for r in qlora_results:

    s = script_stats(r["prediction"])

    stats.append(s)


print("QLoRA TEST RESULTS")
print("=" * 60)

for key in [
    "length",
    "tigrinya_pct",
    "latin_pct",
    "other_pct",
]:

    values = [x[key] for x in stats]

    print(
        f"{key:20s}: "
        f"mean={np.mean(values):.2f}, "
        f"median={np.median(values):.2f}, "
        f"std={np.std(values):.2f}"
    )

QLoRA TEST RESULTS
length              : mean=27.27, median=21.00, std=20.05
tigrinya_pct        : mean=93.63, median=100.00, std=9.18
latin_pct           : mean=0.24, median=0.00, std=4.10
other_pct           : mean=6.13, median=0.00, std=8.39


In [65]:
preamble_patterns = [
    r"here'?s",
    r"here is",
    r"brief",
    r"caption",
    r"option",
    r"the image",
    r"based on",
]


def has_preamble(text):

    text_lower = text.lower()

    return any(
        re.search(pattern, text_lower)
        for pattern in preamble_patterns
    )


preamble_count = sum(
    has_preamble(r["prediction"])
    for r in qlora_results
)

print(
    "Preamble rate:",
    f"{100 * preamble_count / len(qlora_results):.2f}%"
)

Preamble rate: 0.00%


In [66]:
def repetition_score(text):

    words = text.split()

    if len(words) < 5:
        return 0.0

    counts = Counter(words)

    repeated = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated / len(words)


repetition_scores = [
    repetition_score(r["prediction"])
    for r in qlora_results
]

print(
    "Mean repetition score:",
    f"{np.mean(repetition_scores):.4f}"
)

print(
    "Examples with repetition score > 0.30:",
    sum(
        x > 0.30
        for x in repetition_scores
    )
)

Mean repetition score: 0.1110
Examples with repetition score > 0.30: 40


In [67]:
import sacrebleu
from rouge_score import rouge_scorer

# Your already-extracted lists
references = [r["reference"] for r in qlora_results]
predictions = [r["prediction"] for r in qlora_results]


# --------------------------------------------------
# BLEU
# --------------------------------------------------
def calculate_bleu(references, predictions):
    return sacrebleu.corpus_bleu(
        predictions,
        [references]
    ).score


# --------------------------------------------------
# chrF
# --------------------------------------------------
def calculate_chrf(references, predictions):
    return sacrebleu.corpus_chrf(
        predictions,
        [references]
    ).score


# --------------------------------------------------
# ROUGE-L
# --------------------------------------------------
def calculate_rouge_l(references, predictions):
    scorer = rouge_scorer.RougeScorer(
        ["rougeL"],
        use_stemmer=False
    )

    scores = []

    for reference, prediction in zip(references, predictions):
        score = scorer.score(reference, prediction)["rougeL"].fmeasure
        scores.append(score)

    return sum(scores) / len(scores)


# --------------------------------------------------
# Calculate QLoRA metrics
# --------------------------------------------------
qlora_metrics = {
    "BLEU": calculate_bleu(references, predictions),
    "chrF": calculate_chrf(references, predictions),
    "ROUGE-L": calculate_rouge_l(references, predictions),
}


print("QLoRA RAW TEST METRICS")
print("=" * 50)

for metric, value in qlora_metrics.items():
    print(f"{metric:10s}: {value:.6f}")

QLoRA RAW TEST METRICS
BLEU      : 2.143814
chrF      : 9.720557
ROUGE-L   : 0.022801


## 6. Text-Only QLoRA Ablation

This ablation isolates the contribution of the image: the model receives only the Tigrinya news title, never the image, during both training and generation. Comparing this to the multimodal model (Section 5) tests whether the image contributes beyond adapting Gemma to Tigrinya generation from titles alone.

In [68]:
def make_text_only_training_messages(example):
    return [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "ዕላማ ዜና ተመርኲስካ፣ "
                        "ነቲ ኣርእስቲ ዜና ዝምልከት "
                        "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                        f"ኣርእስቲ ዜና፦ {example['title']}"
                    )
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {"type": "text", "text": example["caption"]}
            ],
        },
    ]

In [69]:
from dataclasses import dataclass
import torch

@dataclass
class Gemma3TextOnlyCollator:
    processor: object

    def __call__(self, examples):
        full_inputs_list = []
        prompt_inputs_list = []

        for example in examples:
            messages = make_text_only_training_messages(example)

            full_inputs = self.processor.apply_chat_template(
                messages,
                add_generation_prompt=False,
                tokenize=True,
                return_tensors="pt",
                return_dict=True,
            )

            prompt_inputs = self.processor.apply_chat_template(
                messages[:1],
                add_generation_prompt=True,
                tokenize=True,
                return_tensors="pt",
                return_dict=True,
            )

            full_inputs_list.append(full_inputs)
            prompt_inputs_list.append(prompt_inputs)

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [x["input_ids"][0] for x in full_inputs_list],
            batch_first=True,
            padding_value=self.processor.tokenizer.pad_token_id,
        )

        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [x["attention_mask"][0] for x in full_inputs_list],
            batch_first=True,
            padding_value=0,
        )

        token_type_ids = torch.nn.utils.rnn.pad_sequence(
            [x["token_type_ids"][0] for x in full_inputs_list],
            batch_first=True,
            padding_value=0,
        )

        labels = []

        for full_inputs, prompt_inputs in zip(
            full_inputs_list, prompt_inputs_list
        ):
            ids = full_inputs["input_ids"][0].clone()

            prompt_length = prompt_inputs["input_ids"].shape[1]

            ids[:prompt_length] = -100

            labels.append(ids)

        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "token_type_ids": token_type_ids,
            "labels": labels,
        }

In [70]:
from transformers import BitsAndBytesConfig, Gemma3ForConditionalGeneration
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

text_model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

text_model.config.use_cache = False

text_model = prepare_model_for_kbit_training(text_model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)

text_model = get_peft_model(text_model, lora_config)

text_model.print_trainable_parameters()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

trainable params: 32,788,480 || all params: 4,332,867,952 || trainable%: 0.7567


In [71]:
from torch.optim import AdamW

optimizer = AdamW(
    text_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_epochs = NUM_EPOCHS_ABLATION
gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS
max_grad_norm = MAX_GRAD_NORM

In [73]:
text_collator = Gemma3TextOnlyCollator(processor)

In [74]:
text_train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=text_collator,
    num_workers=0,
)

text_val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=text_collator,
    num_workers=0,
)


In [75]:
import os
import time
import torch

text_model.train()

best_val_loss = float("inf")
best_epoch = None

text_output_dir = "/content/gemma3_tigrinya_text_only_qlora"
os.makedirs(text_output_dir, exist_ok=True)

for epoch in range(num_epochs):

    epoch_start = time.time()

    # -------------------------
    # TRAIN
    # -------------------------
    text_model.train()

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(text_train_loader):

        batch = {
            k: v.to(text_model.device)
            for k, v in batch.items()
        }

        outputs = text_model(**batch)

        loss = outputs.loss
        loss_for_backward = loss / gradient_accumulation_steps

        loss_for_backward.backward()

        running_loss += loss.item()

        if (
            (step + 1) % gradient_accumulation_steps == 0
            or (step + 1) == len(text_train_loader)
        ):

            torch.nn.utils.clip_grad_norm_(
                text_model.parameters(),
                max_grad_norm
            )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

            optimizer_steps += 1

    train_loss = running_loss / len(text_train_loader)

    # -------------------------
    # VALIDATION
    # -------------------------
    text_model.eval()

    val_loss_total = 0.0

    with torch.no_grad():

        for batch in text_val_loader:

            batch = {
                k: v.to(text_model.device)
                for k, v in batch.items()
            }

            outputs = text_model(**batch)

            val_loss_total += outputs.loss.item()

    val_loss = val_loss_total / len(text_val_loader)

    elapsed = (time.time() - epoch_start) / 60

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"time: {elapsed:.2f} min"
    )

    # -------------------------
    # SAVE BEST MODEL
    # -------------------------
    if val_loss < best_val_loss:

        best_val_loss = val_loss
        best_epoch = epoch + 1

        best_dir = os.path.join(
            text_output_dir,
            "best_adapter"
        )

        text_model.save_pretrained(best_dir)
        processor.save_pretrained(best_dir)

        print(f"Saved best adapter → {best_dir}")

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation loss:", best_val_loss)

Epoch 1/1 | train loss: 2.8332 | val loss: 2.5959 | time: 4.13 min
Saved best adapter → /content/gemma3_tigrinya_text_only_qlora/best_adapter

Training complete.
Best epoch: 1
Best validation loss: 2.5959174529711406


In [ ]:
from peft import PeftModel

base_text_model = Gemma3ForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

text_best_model = PeftModel.from_pretrained(
    base_text_model,
    "/content/gemma3_tigrinya_text_only_qlora/best_adapter"
)

text_best_model.eval()

In [77]:
def generate_text_only_qlora(title, max_new_tokens=80):

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "ዕላማ ዜና ተመርኲስካ፣ "
                        "ነቲ ኣርእስቲ ዜና ዝምልከት "
                        "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                        f"ኣርእስቲ ዜና፦ {title}"
                    )
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(text_best_model.device)

    with torch.inference_mode():
        output = text_best_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    generated_tokens = output[
        :, inputs["input_ids"].shape[1]:
    ]

    return processor.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )

In [78]:
import json
import time

text_only_results = []

start = time.time()

for i, example in enumerate(test_ds):

    prediction = generate_text_only_qlora(
        example["title"]
    )

    text_only_results.append({
        "test_index": i,
        "title": example["title"],
        "reference": example["caption"],
        "prediction": prediction,
        "news_source": example["news_source"],
    })

    if (i + 1) % 25 == 0:
        print(f"Generated {i+1}/{len(test_ds)}")

elapsed = (time.time() - start) / 60

print(f"\nCompleted in {elapsed:.2f} minutes")

with open(
    "/content/lamun_tigrinya_text_only_qlora_test_predictions.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        text_only_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved predictions.")

Generated 25/288
Generated 50/288
Generated 75/288
Generated 100/288
Generated 125/288
Generated 150/288
Generated 175/288
Generated 200/288
Generated 225/288
Generated 250/288
Generated 275/288

Completed in 2.91 minutes
Saved predictions.


In [79]:
import re
import numpy as np
import sacrebleu
from rouge_score import rouge_scorer

references_text = [
    r["reference"]
    for r in text_only_results
]

predictions_text = [
    r["prediction"]
    for r in text_only_results
]


# ==============================
# Metrics
# ==============================

bleu = sacrebleu.corpus_bleu(
    predictions_text,
    [references_text]
).score

chrf = sacrebleu.corpus_chrf(
    predictions_text,
    [references_text]
).score

rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

rouge_scores = [
    rouge.score(ref, pred)["rougeL"].fmeasure
    for ref, pred in zip(
        references_text,
        predictions_text
    )
]

rouge_l = np.mean(rouge_scores)


# ==============================
# Diagnostics
# ==============================

TIGRINYA_RANGE = re.compile(r"[\u1200-\u137F]")
LATIN_RANGE = re.compile(r"[A-Za-z]")

def character_stats(text):

    total = len(text)

    if total == 0:
        return 0, 0, 0

    tigrinya = len(TIGRINYA_RANGE.findall(text))
    latin = len(LATIN_RANGE.findall(text))
    other = total - tigrinya - latin

    return (
        100 * tigrinya / total,
        100 * latin / total,
        100 * other / total,
    )


lengths = [
    len(pred)
    for pred in predictions_text
]

stats = [
    character_stats(pred)
    for pred in predictions_text
]

tigrinya_pct = [x[0] for x in stats]
latin_pct = [x[1] for x in stats]
other_pct = [x[2] for x in stats]


# ==============================
# Repetition
# ==============================

def repetition_score(text):

    words = text.split()

    if len(words) == 0:
        return 0.0

    counts = {}

    for word in words:
        counts[word] = counts.get(word, 0) + 1

    repeated_tokens = sum(
        count
        for count in counts.values()
        if count > 1
    )

    return repeated_tokens / len(words)


repetition_scores = [
    repetition_score(pred)
    for pred in predictions_text
]


# ==============================
# Print results
# ==============================

print("TEXT-ONLY QLoRA TEST RESULTS")
print("=" * 60)

print(f"BLEU      : {bleu:.6f}")
print(f"chrF      : {chrf:.6f}")
print(f"ROUGE-L   : {rouge_l:.6f}")

print()

print(
    f"length              : "
    f"mean={np.mean(lengths):.2f}, "
    f"median={np.median(lengths):.2f}, "
    f"std={np.std(lengths):.2f}"
)

print(
    f"tigrinya_pct        : "
    f"mean={np.mean(tigrinya_pct):.2f}, "
    f"median={np.median(tigrinya_pct):.2f}, "
    f"std={np.std(tigrinya_pct):.2f}"
)

print(
    f"latin_pct           : "
    f"mean={np.mean(latin_pct):.2f}, "
    f"median={np.median(latin_pct):.2f}, "
    f"std={np.std(latin_pct):.2f}"
)

print(
    f"other_pct           : "
    f"mean={np.mean(other_pct):.2f}, "
    f"median={np.median(other_pct):.2f}, "
    f"std={np.std(other_pct):.2f}"
)

print(
    f"\nMean repetition score: "
    f"{np.mean(repetition_scores):.4f}"
)

print(
    "Examples with repetition score > 0.30:",
    sum(x > 0.30 for x in repetition_scores)
)

TEXT-ONLY QLoRA TEST RESULTS
BLEU      : 1.230344
chrF      : 7.793986
ROUGE-L   : 0.002315

length              : mean=20.80, median=18.00, std=16.02
tigrinya_pct        : mean=82.01, median=83.21, std=11.07
latin_pct           : mean=0.00, median=0.00, std=0.00
other_pct           : mean=17.99, median=16.79, std=11.07

Mean repetition score: 0.0478
Examples with repetition score > 0.30: 19


## 7. Training-Data-Size Ablation

We retrain the multimodal QLoRA model from scratch on random subsets of 100, 500, 1,000, and 2,440 (full) training examples, using a fixed seed for subset sampling, and evaluate each resulting model on the same held-out test set.

In [80]:
import random

# Reproducible sampling
random.seed(42)

# train_ds currently contains 2,440 examples
n_total = len(train_ds)

print("Full training set:", n_total)

subset_sizes = [100, 500, 1000, 2440]

subset_indices = {}

all_indices = list(range(n_total))

for n in subset_sizes:
    if n == n_total:
        subset_indices[n] = all_indices.copy()
    else:
        subset_indices[n] = random.sample(all_indices, n)

for n in subset_sizes:
    print(f"{n} examples:", len(subset_indices[n]))

Full training set: 2440
100 examples: 100
500 examples: 500
1000 examples: 1000
2440 examples: 2440


In [81]:
from torch.utils.data import DataLoader

efficiency_loaders = {}

for n in subset_sizes:

    subset = train_ds.select(subset_indices[n])

    loader = DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collator,
        num_workers=0,
    )

    efficiency_loaders[n] = loader

    print(
        f"{n} examples → "
        f"{len(loader)} batches"
    )

100 examples → 50 batches
500 examples → 250 batches
1000 examples → 500 batches
2440 examples → 1220 batches


In [82]:
def create_qlora_model():

    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model.config.use_cache = False

    model = prepare_model_for_kbit_training(model)

    model = get_peft_model(
        model,
        lora_config
    )

    return model

In [83]:
import os
import time
import torch
from torch.optim import AdamW


def train_efficiency_model(
    train_loader,
    val_loader,
    n_examples,
    output_root="/content/gemma3_tigrinya_data_efficiency",
):

    print("\n" + "=" * 70)
    print(f"TRAINING WITH {n_examples} EXAMPLES")
    print("=" * 70)

    # Fresh base model
    model = create_qlora_model()

    model.train()

    optimizer = AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS
    max_grad_norm = MAX_GRAD_NORM

    best_val_loss = float("inf")

    output_dir = os.path.join(
        output_root,
        f"{n_examples}_examples",
        "best_adapter"
    )

    os.makedirs(output_dir, exist_ok=True)

    # -----------------------------------------
    # ONE EPOCH
    # -----------------------------------------

    start_time = time.time()

    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0

    for step, batch in enumerate(train_loader):

        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        loss = outputs.loss

        running_loss += loss.item()

        loss_for_backward = (
            loss / gradient_accumulation_steps
        )

        loss_for_backward.backward()

        if (
            (step + 1) % gradient_accumulation_steps == 0
            or (step + 1) == len(train_loader)
        ):

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_grad_norm
            )

            optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

    train_loss = (
        running_loss /
        len(train_loader)
    )

    # -----------------------------------------
    # VALIDATION
    # -----------------------------------------

    model.eval()

    val_loss_total = 0.0

    with torch.no_grad():

        for batch in val_loader:

            batch = {
                k: v.to(model.device)
                for k, v in batch.items()
            }

            outputs = model(**batch)

            val_loss_total += outputs.loss.item()

    val_loss = (
        val_loss_total /
        len(val_loader)
    )

    elapsed = (
        time.time() - start_time
    ) / 60

    # -----------------------------------------
    # SAVE
    # -----------------------------------------

    model.save_pretrained(output_dir)
    processor.save_pretrained(output_dir)

    print(
        f"\n{n_examples} examples finished"
    )

    print(
        f"Train loss: {train_loss:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f}"
    )

    print(
        f"Time:       {elapsed:.2f} min"
    )

    print(
        f"Saved:      {output_dir}"
    )

    # Free GPU memory
    del model
    del optimizer

    torch.cuda.empty_cache()

    return {
        "n_examples": n_examples,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "time_minutes": elapsed,
        "adapter_path": output_dir,
    }

In [84]:
efficiency_results = []

for n in subset_sizes:

    result = train_efficiency_model(
        train_loader=efficiency_loaders[n],
        val_loader=val_loader,
        n_examples=n,
    )

    efficiency_results.append(result)

    print("\nCurrent results:")
    for r in efficiency_results:
        print(
            f"{r['n_examples']:4d} examples | "
            f"train={r['train_loss']:.4f} | "
            f"val={r['val_loss']:.4f}"
        )


TRAINING WITH 100 EXAMPLES


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


100 examples finished
Train loss: 4.0135
Val loss:   3.3082
Time:       1.67 min
Saved:      /content/gemma3_tigrinya_data_efficiency/100_examples/best_adapter

Current results:
 100 examples | train=4.0135 | val=3.3082

TRAINING WITH 500 EXAMPLES


...


1000 examples finished
Train loss: 3.0434
Val loss:   2.7144
Time:       10.56 min
Saved:      /content/gemma3_tigrinya_data_efficiency/1000_examples/best_adapter

Current results:
 100 examples | train=4.0135 | val=3.3082
 500 examples | train=3.2565 | val=2.8861
1000 examples | train=3.0434 | val=2.7144

TRAINING WITH 2440 EXAMPLES


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]


2440 examples finished
Train loss: 2.7525
Val loss:   2.5008
Time:       24.75 min
Saved:      /content/gemma3_tigrinya_data_efficiency/2440_examples/best_adapter

Current results:
 100 examples | train=4.0135 | val=3.3082
 500 examples | train=3.2565 | val=2.8861
1000 examples | train=3.0434 | val=2.7144
2440 examples | train=2.7525 | val=2.5008


In [85]:
import json

with open(
    "/content/gemma3_tigrinya_data_efficiency_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        efficiency_results,
        f,
        indent=2
    )

print("Results saved.")

Results saved.


In [86]:
import pandas as pd

efficiency_df = pd.DataFrame(
    efficiency_results
)

display(efficiency_df)

,n_examples,train_loss,val_loss,time_minutes,adapter_path
0,100,4.013549,3.308153,1.671320,/content/gemma3_tigrinya_data_efficiency/100_e...
1,500,3.256542,2.886061,5.615969,/content/gemma3_tigrinya_data_efficiency/500_e...
2,1000,3.043445,2.714365,10.557276,/content/gemma3_tigrinya_data_efficiency/1000_...
3,2440,2.752511,2.500836,24.747021,/content/gemma3_tigrinya_data_efficiency/2440_...


### 7.1 Evaluation of each data-size model on the test set

In [87]:
import os
import json
import time
import torch
from peft import PeftModel

data_efficiency_predictions = {}

for n in [100, 500, 1000, 2440]:

    print("\n" + "=" * 70)
    print(f"EVALUATING {n} TRAINING EXAMPLES")
    print("=" * 70)

    adapter_path = (
        f"/content/gemma3_tigrinya_data_efficiency/"
        f"{n}_examples/best_adapter"
    )

    # Fresh base model
    base_model = Gemma3ForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path
    )

    model.eval()

    results = []

    start = time.time()

    for i, example in enumerate(test_ds):

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": example["image"]
                    },
                    {
                        "type": "text",
                        "text": (
                            "ዕላማ ዜና ተመርኲስካ፣ "
                            "ነቲ ስእሊ ዝገልጽ "
                            "ሓጺር መግለጺ ብትግርኛ ጽሓፍ።\n\n"
                            f"ኣርእስቲ ዜና፦ {example['title']}"
                        )
                    }
                ]
            }
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)

        with torch.inference_mode():

            output = model.generate(
                **inputs,
                max_new_tokens=80,
                do_sample=False,
            )

        generated_tokens = output[
            :,
            inputs["input_ids"].shape[1]:
        ]

        prediction = processor.decode(
            generated_tokens[0],
            skip_special_tokens=True
        )

        results.append({
            "test_index": i,
            "title": example["title"],
            "reference": example["caption"],
            "prediction": prediction,
            "news_source": example["news_source"],
        })

        if (i + 1) % 50 == 0:
            print(
                f"Generated {i+1}/{len(test_ds)}"
            )

    elapsed = (time.time() - start) / 60

    data_efficiency_predictions[n] = results

    output_file = (
        f"/content/"
        f"lamun_tigrinya_qlora_{n}_examples_predictions.json"
    )

    with open(
        output_file,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            results,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(
        f"Completed {n} examples in "
        f"{elapsed:.2f} minutes"
    )

    print(
        f"Saved → {output_file}"
    )

    # Free memory
    del model
    del base_model
    torch.cuda.empty_cache()


EVALUATING 100 TRAINING EXAMPLES


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Generated 50/288
Generated 100/288
Generated 150/288
Generated 200/288
Generated 250/288
Completed 100 examples in 7.84 minutes
Saved → /content/lamun_tigrinya_qlora_100_examples_predictions.json

EVALUATING 500 TRAINING EXAMPLES


...

Generated 50/288
Generated 100/288
Generated 150/288
Generated 200/288
Generated 250/288
Completed 1000 examples in 5.51 minutes
Saved → /content/lamun_tigrinya_qlora_1000_examples_predictions.json

EVALUATING 2440 TRAINING EXAMPLES


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Generated 50/288
Generated 100/288
Generated 150/288
Generated 200/288
Generated 250/288
Completed 2440 examples in 5.85 minutes
Saved → /content/lamun_tigrinya_qlora_2440_examples_predictions.json


In [88]:
import json
import re
import numpy as np
import pandas as pd
from sacrebleu import corpus_bleu, corpus_chrf
from rouge_score import rouge_scorer

sizes = [100, 500, 1000, 2440]

def normalize_for_metrics(text):
    """
    Remove obvious English boilerplate only.
    Keep the actual generated Tigrinya content unchanged.
    """
    text = text.strip()

    patterns = [
        r"^Here is (a|the) (short )?(description|caption).*?:\s*",
        r"^Here'?s (a|the) (short )?(description|caption).*?:\s*",
        r"^Description:\s*",
        r"^Caption:\s*",
        r"^Answer:\s*",
    ]

    for pattern in patterns:
        text = re.sub(pattern, "", text, flags=re.IGNORECASE)

    return text.strip()


def tigrinya_pct(text):
    if not text:
        return 0.0

    total = len(text)
    tigrinya = sum(
        0x1200 <= ord(c) <= 0x137F
        for c in text
    )
    return 100 * tigrinya / total


def latin_pct(text):
    if not text:
        return 0.0

    total = len(text)
    latin = sum(
        ("A" <= c <= "Z") or ("a" <= c <= "z")
        for c in text
    )
    return 100 * latin / total


def repetition_score(text):
    """
    Fraction of repeated adjacent bigrams.
    """
    tokens = text.split()

    if len(tokens) < 2:
        return 0.0

    bigrams = list(zip(tokens[:-1], tokens[1:]))

    if not bigrams:
        return 0.0

    unique = len(set(bigrams))
    return 1 - unique / len(bigrams)


rouge = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=False
)

summary = []
all_predictions = {}

for n in sizes:

    path = f"/content/lamun_tigrinya_qlora_{n}_examples_predictions.json"

    with open(path, "r", encoding="utf-8") as f:
        results = json.load(f)

    all_predictions[n] = results

    references = [
        normalize_for_metrics(x["reference"])
        for x in results
    ]

    predictions = [
        normalize_for_metrics(x["prediction"])
        for x in results
    ]

    # BLEU
    bleu = corpus_bleu(
        predictions,
        [references]
    ).score

    # chrF
    chrf = corpus_chrf(
        predictions,
        [references]
    ).score

    # ROUGE-L
    rouge_scores = []

    for pred, ref in zip(predictions, references):
        score = rouge.score(ref, pred)["rougeL"].fmeasure
        rouge_scores.append(score)

    rouge_l = np.mean(rouge_scores)

    # Diagnostics
    lengths = [len(x) for x in predictions]
    tig = [tigrinya_pct(x) for x in predictions]
    latin = [latin_pct(x) for x in predictions]
    repetition = [repetition_score(x) for x in predictions]

    high_rep = sum(r > 0.30 for r in repetition)

    summary.append({
        "training_examples": n,
        "BLEU": bleu,
        "chrF": chrf,
        "ROUGE-L": rouge_l,
        "mean_length": np.mean(lengths),
        "median_length": np.median(lengths),
        "Tigrinya_%": np.mean(tig),
        "Latin_%": np.mean(latin),
        "mean_repetition": np.mean(repetition),
        "high_repetition_count": high_rep,
    })

results_df = pd.DataFrame(summary)

print("\n" + "=" * 90)
print("DATA-EFFICIENCY TEST RESULTS")
print("=" * 90)

display(results_df.round(4))


DATA-EFFICIENCY TEST RESULTS


,training_examples,BLEU,chrF,ROUGE-L,mean_length,median_length,Tigrinya_%,Latin_%,mean_repetition,high_repetition_count
0,100,1.0110,9.1047,0.0176,59.5694,50.0,78.1389,0.4963,0.2486,107
1,500,1.0252,6.9415,0.0226,32.7465,25.5,71.1162,0.3194,0.1464,62
2,1000,1.6177,9.2757,0.0104,40.6389,34.0,76.4975,0.3194,0.1017,45
3,2440,1.6831,10.1459,0.0156,43.8125,36.0,76.4862,0.1997,0.1230,51


In [89]:
val_losses = {
    100: 3.293035,
    500: 2.892499,
    1000: 2.721818,
    2440: 2.505815,
}

results_df["val_loss"] = results_df["training_examples"].map(val_losses)

results_df = results_df[
    [
        "training_examples",
        "val_loss",
        "BLEU",
        "chrF",
        "ROUGE-L",
        "Tigrinya_%",
        "mean_repetition",
        "high_repetition_count",
    ]
]

display(results_df.round(4))

,training_examples,val_loss,BLEU,chrF,ROUGE-L,Tigrinya_%,mean_repetition,high_repetition_count
0,100,3.2930,1.0110,9.1047,0.0176,78.1389,0.2486,107
1,500,2.8925,1.0252,6.9415,0.0226,71.1162,0.1464,62
2,1000,2.7218,1.6177,9.2757,0.0104,76.4975,0.1017,45
3,2440,2.5058,1.6831,10.1459,0.0156,76.4862,0.1230,51


In [90]:
# ============================================================
# Final Consolidated Results
# ============================================================

import pandas as pd

# Main multimodal vs. text-only comparison
main_results = pd.DataFrame([
    {
        "Experiment": "Zero-shot Image + Title",
        "BLEU": 0.094597,
        "chrF": 3.882567,
        "ROUGE-L": 0.001657,
    },
    {
        "Experiment": "Text-only QLoRA",
        "BLEU": 1.230344,
        "chrF": 7.793986,
        "ROUGE-L": 0.002315,
    },
    {
        "Experiment": "Multimodal QLoRA",
        "BLEU": 2.143814,
        "chrF": 9.720557,
        "ROUGE-L": 0.022801,
    },
])

print("Main Experimental Comparison")
display(main_results.round(4))

Main Experimental Comparison


,Experiment,BLEU,chrF,ROUGE-L
0,Zero-shot Image + Title,0.0946,3.8826,0.0017
1,Text-only QLoRA,1.2303,7.7940,0.0023
2,Multimodal QLoRA,2.1438,9.7206,0.0228


In [91]:
# ============================================================
# Data-Size Ablation
# ============================================================

data_size_results = results_df[
    [
        "training_examples",
        "val_loss",
        "BLEU",
        "chrF",
        "ROUGE-L",
        "Tigrinya_%",
        "mean_repetition",
        "high_repetition_count",
    ]
].copy()

data_size_results.columns = [
    "Training Examples",
    "Validation Loss",
    "BLEU",
    "chrF",
    "ROUGE-L",
    "Tigrinya (%)",
    "Mean Repetition",
    "High Repetition (>0.30)",
]

print("Data-Size Ablation")
display(data_size_results.round(4))

Data-Size Ablation


,Training Examples,Validation Loss,BLEU,chrF,ROUGE-L,Tigrinya (%),Mean Repetition,High Repetition (>0.30)
0,100,3.2930,1.0110,9.1047,0.0176,78.1389,0.2486,107
1,500,2.8925,1.0252,6.9415,0.0226,71.1162,0.1464,62
2,1000,2.7218,1.6177,9.2757,0.0104,76.4975,0.1017,45
3,2440,2.5058,1.6831,10.1459,0.0156,76.4862,0.1230,51


### Key Findings

- Multimodal QLoRA substantially improves Tigrinya generation compared with the zero-shot image+title baseline.
- The multimodal model also outperforms the text-only QLoRA ablation across BLEU, chrF, and ROUGE-L, indicating that incorporating the image provides useful information beyond language-only adaptation.
- Increasing the training set from 100 to 2,440 examples consistently reduces validation loss, while test-set lexical metrics vary across intermediate data sizes.
- The full 2,440-example setting achieves the strongest overall test performance among the evaluated data sizes.
- Despite improved Tigrinya generation, qualitative inspection indicates that visual grounding and factual consistency remain challenging.

## Limitations

- The Tigrinya subset of LaMuN is small (3,028 examples / 1,768 unique articles),
  limiting statistical power, especially in the low-data regime (100/500 examples).
- Reference captions are news captions, not pure visual descriptions — they are not
  always fully inferable from the image alone, which caps achievable scores on
  lexical-overlap metrics regardless of model quality.
- BLEU/chrF/ROUGE-L measure lexical overlap with a single reference and do not
  directly measure visual grounding or factual correctness.
- Qualitative inspection indicates the adapted model can produce fluent Tigrinya
  while still generating content that is insufficiently grounded in the image or
  reference context — i.e. language quality and multimodal grounding are not the
  same thing, and this gap is not captured by the automatic metrics above.
- Each data-size condition was trained once (one epoch, one seed) rather than
  averaged over multiple runs, so the non-monotonic trend across data sizes should
  be read with that variance in mind.

See `README.md` for the polished results summary and findings.
